In [1]:
"""
cts_model.py — Physics-Grounded CTS Prediction
================================================
Architecture decision: NO graph compression.

CTS-Bench (2026) proves that graph compression destroys skew signal.
GAN-CTS (TCAD 2022) uses ResNet50 spatial features + MLP, not a GNN.
Kahng UCSD (c300): HPWL, aspect ratio, nonuniformity are the key features.

This model:
  1. Computes physics-grounded features from FF positions (no compression)
  2. Uses per-placement normalization for targets (not global)
  3. Predicts relative to placement mean: "how does THIS config compare to
     average for this design?" — directly learnable from knob × geometry
  4. Explicit physics interaction terms for each task's head

Per-task physics:
  SKEW:  skew = max_path_imbalance = hard to balance paths
         key: max_skip_dist / buf_dist, cluster_dia / kNN4_dist
  POWER: power ≈ k1×WL + k2×n_ff/cluster_size
         key: cluster_size, cluster_dia, HPWL, n_ff × cap
  WL:    WL ≈ 1.2 × HPWL × f(cluster_params)
         key: cluster_dia / HPWL, max_wire, cluster_size × kNN
"""

import os, math, random
from collections import defaultdict

import numpy as np
import pandas as pd
import scipy.sparse as sp

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

from cts_features import (
    build_ff_features, build_sgc_features, FEAT_DIM,
    IDX_HPWL, IDX_SKIP_MAX, IDX_SKIP_MEAN, IDX_SKIP_P90,
    IDX_KNN4_MEAN, IDX_KNN4_P90, IDX_CENTROID_DIST,
    IDX_SPREAD_ASYM, IDX_N_FF, IDX_CAP_MEAN,
)

PT_DIR   = "processed_graphs/"
CSV_DIR  = "dataset_with_def/"
CSV_PATH = os.path.join(CSV_DIR, "unified_manifest_normalized.csv")

# ─────────────────────────────────────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────────────────────────────────────
NUM_KNOBS  = 4
SGC_DIM    = 27   # 9 features × 3 (1 + 2 hops)
HIDDEN     = 128
EPOCHS     = 300
LR         = 1e-3

def get_k(n_ff):
    return max(16, min(64, n_ff // 8))


# ─────────────────────────────────────────────────────────────────────────────
# 1. LOADING UTILITIES
# ─────────────────────────────────────────────────────────────────────────────

def extract_ff_info(data):
    X      = data['X']
    ff_idx = (X[:, 10] == 1.0).nonzero(as_tuple=True)[0]
    return ff_idx, X[ff_idx]


def sparse_csr_to_local_edges(A_csr, ff_idx_np):
    try:
        A_coo = A_csr.to_sparse_coo().coalesce()
        idx   = A_coo.indices().numpy()
        if idx.shape[1] == 0:
            return np.zeros((0, 2), dtype=np.int64)
    except Exception:
        return np.zeros((0, 2), dtype=np.int64)
    g2l   = {int(g): l for l, g in enumerate(ff_idx_np)}
    pairs = [[g2l[int(s)], g2l[int(d)]]
             for s, d in zip(idx[0], idx[1])
             if int(s) in g2l and int(d) in g2l]
    return np.array(pairs, dtype=np.int64) if pairs \
           else np.zeros((0, 2), dtype=np.int64)


# ─────────────────────────────────────────────────────────────────────────────
# 2. GLOBAL NORMALIZATION (features only, not targets)
# ─────────────────────────────────────────────────────────────────────────────

def compute_global_stats(matrices):
    all_X = np.concatenate(matrices, axis=0)
    p5    = np.percentile(all_X,  5, axis=0)
    p95   = np.percentile(all_X, 95, axis=0)
    clipped = np.clip(all_X, p5, p95)
    mu  = clipped.mean(axis=0).astype(np.float32)
    sig = clipped.std(axis=0).astype(np.float32)
    return mu, np.where(sig < 1e-8, 1.0, sig)

def apply_norm(X, mu, sig):
    return (X - mu) / sig


# ─────────────────────────────────────────────────────────────────────────────
# 3. DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_dataset(pt_dir, csv_path):
    df       = pd.read_csv(csv_path)
    df_by_id = {pid: grp for pid, grp in df.groupby('placement_id')}
    pt_files = sorted([f for f in os.listdir(pt_dir) if f.endswith('.pt')])
    print(f"Found {len(pt_files)} .pt files")
    raw = []

    for pt_file in pt_files:
        pid   = pt_file.replace('.pt', '')
        if pid not in df_by_id: continue
        group = df_by_id[pid]
        row0  = group.iloc[0]
        design= str(row0.get('design_name', pid.split('_')[0]))
        print(f"\n{'─'*48}\nLoading: {pid}  ({design})")

        try:
            data = torch.load(os.path.join(pt_dir, pt_file), map_location='cpu')
        except Exception as e:
            print(f"  SKIP: {e}"); continue

        X_all     = data['X']
        num_nodes = data['num_nodes']
        ff_idx, ff_X_raw = extract_ff_info(data)
        n_ff = len(ff_idx)
        if n_ff < 8:
            print(f"  SKIP: only {n_ff} FFs"); continue

        ff_idx_np = ff_idx.numpy()
        ff_skip_local = sparse_csr_to_local_edges(data['A_skip_csr'], ff_idx_np)

        # Physics features (no graph compression)
        ff_feats  = build_ff_features(ff_X_raw, ff_skip_local, X_all, num_nodes)

        # SGC features for each FF (used for pooling if needed)
        ff_X_smooth = build_sgc_features(ff_X_raw, ff_skip_local)

        # Per-placement normalization of targets
        # Reason: global z-scores conflate design-type effects with CTS effects.
        # We want to predict "how does this config compare to average for THIS placement?"
        cts_runs_raw = []
        for _, row in group.iterrows():
            knobs = torch.tensor([
                row['z_cts_max_wire'], row['z_cts_buf_dist'],
                row['z_cts_cluster_size'], row['z_cts_cluster_dia'],
            ], dtype=torch.float32)
            cts_runs_raw.append({
                'knobs':  knobs,
                'skew':   float(row['skew_setup']),
                'power':  float(row['power_total']),
                'wl':     float(row['wirelength']),
            })

        if not cts_runs_raw: continue

        # Per-placement normalize: z = (x - mean) / max(std, floor)
        cts_runs = []
        for task in ['skew', 'power', 'wl']:
            vals  = np.array([r[task] for r in cts_runs_raw], np.float32)
            mu_t  = vals.mean()
            sig_t = vals.std()
            # Floor: at least 1% of mean magnitude, min 1e-4
            # Prevents explosion when a metric barely varies within a placement
            sig_t = max(sig_t, max(abs(mu_t) * 0.01, 1e-4))
            for r in cts_runs_raw:
                r[f'z_{task}'] = (r[task] - mu_t) / sig_t

        for r in cts_runs_raw:
            cts_runs.append({
                'knobs':   r['knobs'],
                'targets': {
                    'skew':  torch.tensor([r['z_skew']],  dtype=torch.float32),
                    'power': torch.tensor([r['z_power']], dtype=torch.float32),
                    'wl':    torch.tensor([r['z_wl']],    dtype=torch.float32),
                }
            })

        raw.append({
            'placement_id':  pid,
            'design_name':   design,
            'ff_feats':      ff_feats,       # [50] physics features
            'ff_X_smooth':   ff_X_smooth,    # [n_ff, 27] SGC features
            'ff_skip_local': ff_skip_local,
            'num_nodes':     num_nodes,
            'n_ff':          n_ff,
            'k':             get_k(n_ff),
            'cts_runs':      cts_runs,
        })
        print(f"  OK  n_ff={n_ff}  skip={len(ff_skip_local)}  runs={len(cts_runs)}")

    if not raw: return []

    print(f"\nNormalizing features across {len(raw)} placements...")
    mu_ff,  sig_ff  = compute_global_stats([p['ff_feats'].reshape(1,-1) for p in raw])
    mu_sgc, sig_sgc = compute_global_stats([p['ff_X_smooth'] for p in raw])
    print(f"  FF feats: mu=[{mu_ff.min():.2f},{mu_ff.max():.2f}]  "
          f"sig=[{sig_ff.min():.2f},{sig_ff.max():.2f}]")
    print(f"  SGC:      mu=[{mu_sgc.min():.2f},{mu_sgc.max():.2f}]  "
          f"sig=[{sig_sgc.min():.2f},{sig_sgc.max():.2f}]")

    placements = []
    for p in raw:
        placements.append({
            **p,
            'ff_feats': torch.tensor(
                apply_norm(p['ff_feats'].reshape(1,-1), mu_ff, sig_ff).flatten(),
                dtype=torch.float32),
            'ff_X_smooth': torch.tensor(
                apply_norm(p['ff_X_smooth'], mu_sgc, sig_sgc),
                dtype=torch.float32),
            'ff_skip_local': torch.tensor(p['ff_skip_local'], dtype=torch.long),
        })

    print(f"\n{'═'*48}\nLoaded {len(placements)} placements")
    by_d = defaultdict(int)
    for p in placements: by_d[p['design_name']] += 1
    for d, c in sorted(by_d.items()): print(f"  {d}: {c}")
    return placements, mu_ff, sig_ff, mu_sgc, sig_sgc


# ─────────────────────────────────────────────────────────────────────────────
# 4. MODEL
# ─────────────────────────────────────────────────────────────────────────────

class CTSPredictor(nn.Module):
    """
    Physics-grounded predictor using BOTH:
    1. Global placement features (50-dim) — for WL and power global structure
    2. Per-FF SGC features (n_ff × 27) — pooled with order statistics for skew

    Order statistics pooling for skew:
      - Sort FFs by their skip-path distance (launch-capture pairs)
      - Top-k pool: take mean of top 10% highest-distance FFs
      - This preserves worst-case timing path signal that averaging destroys

    This directly addresses the core failure: skew is determined by the
    WORST path, not the average path. Mean pooling destroys the signal.
    

    Input for each task:
      [ff_feats(50) | knobs(4) | task_interactions(N)]

    Interactions are physics-derived products of knob values × design features.
    These encode the specific knob-geometry coupling for each task, which the
    network cannot learn from 140 samples alone.

    SKEW interactions (7):
      - buf_dist / (skip_max + eps)     : ratio determines buffer stages per path
      - cluster_dia / (knn4_mean + eps) : cluster coverage vs local FF spacing
      - cluster_dia * centroid_dist     : cluster scope vs launch-capture gap
      - buf_dist * skip_max             : buffer density × worst path
      - cluster_sz * knn4_mean          : cluster granularity × local density
      - (1 / buf_dist+1) * skip_p90    : buffer budget vs 90th pct path
      - spread_asym * cluster_dia       : asymmetric design × cluster scope

    POWER interactions (5):
      - (1/cluster_size) * n_ff_proxy   : n_buffers ∝ n_ff / cluster_size
      - cluster_dia * hpwl              : cluster scope × tree extent → WL
      - max_wire * hpwl                 : wire budget × tree extent
      - buf_dist * cap_mean             : buffer density × capacitive load
      - cluster_size * knn4_mean        : cluster size × local density

    WL interactions (5):
      - cluster_dia * hpwl              : cluster scope × tree HPWL
      - cluster_sz / (hpwl + eps)       : cluster granularity per unit area
      - max_wire / (hpwl + eps)         : wire budget relative to tree size
      - cluster_dia * knn4_mean         : cluster size vs FF spacing
      - max_wire * cluster_sz           : max segment × n_segments proxy
    """

    N_SKEW_INT  = 7
    N_POWER_INT = 5
    N_WL_INT    = 5

    def __init__(self, feat_dim=FEAT_DIM, sgc_dim=SGC_DIM,
                 n_knobs=NUM_KNOBS, hidden=HIDDEN):
        super().__init__()
        self.feat_dim = feat_dim
        self.sgc_dim  = sgc_dim

        # Shared encoder for global features (WL, power)
        self.encoder = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden), nn.GELU(),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2), nn.GELU(),
        )
        enc_dim = hidden // 2   # 64

        # Skew-specific encoder for per-FF order statistics
        # Takes top-k pooled SGC features (worst-case paths)
        # top10%_mean + top25%_mean + max = 3 × sgc_dim
        skew_pool_dim = 3 * sgc_dim
        self.skew_encoder = nn.Sequential(
            nn.Linear(skew_pool_dim, hidden // 2),
            nn.LayerNorm(hidden // 2), nn.GELU(),
            nn.Linear(hidden // 2, hidden // 4),
            nn.GELU(),
        )
        skew_enc_dim = hidden // 4   # 32

        # Head inputs
        skew_in  = enc_dim + skew_enc_dim + n_knobs + self.N_SKEW_INT
        power_in = enc_dim + n_knobs + self.N_POWER_INT
        wl_in    = enc_dim + n_knobs + self.N_WL_INT

        # LayerNorm on interaction terms
        self.ln_skew_int  = nn.LayerNorm(self.N_SKEW_INT)
        self.ln_power_int = nn.LayerNorm(self.N_POWER_INT)
        self.ln_wl_int    = nn.LayerNorm(self.N_WL_INT)

        # Task heads
        self.head_skew  = self._make_head(skew_in,  hidden // 2)
        self.head_power = self._make_head(power_in, hidden // 2)
        self.head_wl    = self._make_head(wl_in,    hidden // 2)

    def _make_head(self, in_dim, hid):
        return nn.Sequential(
            nn.Linear(in_dim, hid), nn.GELU(),
            nn.Linear(hid, hid//2), nn.GELU(),
            nn.Linear(hid//2, 1),
        )

    def _get_feat(self, ff_feats, idx):
        """Extract feature at index from normalized feature vector."""
        return ff_feats[idx]

    def order_stats_pool(self, ff_X, ff_skip_local):
        """
        Pool per-FF features using order statistics — preserves worst-case signal.

        For skew prediction we need the TAIL of the distribution, not the mean.
        Skew = worst-case path imbalance → FFs on the longest paths matter most.

        We sort FFs by their skip-path involvement (how many timing paths touch them)
        and pool the top 10%, top 25%, and overall max.
        """
        n_ff = ff_X.shape[0]

        # Weight each FF by how many skip (timing) paths it participates in
        path_weight = torch.zeros(n_ff, device=ff_X.device)
        if ff_skip_local.shape[0] > 0:
            for idx in [ff_skip_local[:, 0], ff_skip_local[:, 1]]:
                path_weight.scatter_add_(0, idx,
                    torch.ones(len(idx), device=ff_X.device))
        # Normalize weights to [0,1]
        path_weight = path_weight / (path_weight.max() + 1e-8)

        # Sort FFs by path involvement (highest first)
        sorted_idx = torch.argsort(path_weight, descending=True)
        ff_sorted  = ff_X[sorted_idx]   # [n_ff, sgc_dim]

        # Top 10% pool (worst-case paths)
        k10  = max(1, n_ff // 10)
        z_top10 = ff_sorted[:k10].mean(dim=0)    # [sgc_dim]

        # Top 25% pool
        k25  = max(1, n_ff // 4)
        z_top25 = ff_sorted[:k25].mean(dim=0)    # [sgc_dim]

        # Max pool (single worst FF)
        z_max = ff_X.max(dim=0)[0]               # [sgc_dim]

        return torch.cat([z_top10, z_top25, z_max])   # [3 * sgc_dim]

    def forward(self, ff_feats, ff_X_smooth, ff_skip_local, knobs_batch):
        """
        ff_feats:      [feat_dim]   — global placement features
        ff_X_smooth:   [n_ff, 27]  — per-FF SGC features
        ff_skip_local: [E, 2]      — timing path edges (local FF indices)
        knobs_batch:   [B, n_knobs]

        Returns: {'skew': [B], 'power': [B], 'wl': [B]}
        """
        B = knobs_batch.shape[0]

        # Global encoder (used for power + WL)
        enc = self.encoder(ff_feats)             # [64]
        enc_exp = enc.unsqueeze(0).expand(B, -1) # [B, 64]

        # Per-FF order statistics for skew (preserves worst-case signal)
        skew_pool = self.order_stats_pool(ff_X_smooth, ff_skip_local)   # [3*27]
        skew_enc  = self.skew_encoder(skew_pool)                        # [32]
        skew_enc_exp = skew_enc.unsqueeze(0).expand(B, -1)              # [B, 32]

        # Extract design features for interaction terms
        hpwl      = self._get_feat(ff_feats, IDX_HPWL)
        skip_max  = self._get_feat(ff_feats, IDX_SKIP_MAX)
        skip_p90  = self._get_feat(ff_feats, IDX_SKIP_P90)
        skip_mean = self._get_feat(ff_feats, IDX_SKIP_MEAN)
        knn4_mean = self._get_feat(ff_feats, IDX_KNN4_MEAN)
        knn4_p90  = self._get_feat(ff_feats, IDX_KNN4_P90)
        centd     = self._get_feat(ff_feats, IDX_CENTROID_DIST)
        sasym     = self._get_feat(ff_feats, IDX_SPREAD_ASYM)
        n_ff_prx  = self._get_feat(ff_feats, IDX_N_FF)
        cap_mean  = self._get_feat(ff_feats, IDX_CAP_MEAN)

        # CTS knobs: [max_wire, buf_dist, cluster_size, cluster_dia]
        max_wire  = knobs_batch[:, 0]
        buf_dist  = knobs_batch[:, 1]
        clust_sz  = knobs_batch[:, 2]
        clust_dia = knobs_batch[:, 3]

        # ── SKEW interaction terms ────────────────────────────────────────
        # Physics: skew ∝ difficulty of equalizing paths
        # buf_dist / path_length = buffer stages per path (more = better balance)
        # cluster_dia / knn4 = how many FFs cluster together (more = better local balance)
        s1 = buf_dist / (skip_max.expand(B) + 0.1)      # buf stages per worst path
        s2 = clust_dia / (knn4_mean.expand(B) + 0.1)    # cluster coverage
        s3 = clust_dia * centd.expand(B)                 # scope vs launch-capture gap
        s4 = buf_dist * skip_max.expand(B)               # buf density × worst path
        s5 = clust_sz * knn4_mean.expand(B)              # granularity × spacing
        s6 = skip_p90.expand(B) / (buf_dist + 0.1)      # 90th pct path / buf budget
        s7 = sasym.expand(B) * clust_dia                 # asymmetry × cluster scope
        skew_int = self.ln_skew_int(
            torch.stack([s1, s2, s3, s4, s5, s6, s7], dim=1))  # [B, 7]

        # ── POWER interaction terms ───────────────────────────────────────
        # Physics: P ≈ k1×WL + k2×n_ff/cluster_size
        # n_buffers ∝ n_ff / cluster_size → power ↑ when cluster_size ↓
        p1 = 1.0 / (clust_sz.abs() + 0.1) * n_ff_prx.expand(B)  # n_buffers proxy
        p2 = clust_dia * hpwl.expand(B)                          # scope × WL proxy
        p3 = max_wire * hpwl.expand(B)                           # budget × WL
        p4 = buf_dist * cap_mean.expand(B)                       # buf density × cap
        p5 = clust_sz * knn4_mean.expand(B)                      # granularity × spacing
        power_int = self.ln_power_int(
            torch.stack([p1, p2, p3, p4, p5], dim=1))  # [B, 5]

        # ── WL interaction terms ──────────────────────────────────────────
        # Physics: WL ≈ 1.2 × HPWL × f(cluster_params)
        # cluster_dia × HPWL: larger clusters span larger fractions of tree
        w1 = clust_dia * hpwl.expand(B)                          # scope × extent
        w2 = clust_sz / (hpwl.expand(B) + 0.1)                  # granularity per area
        w3 = max_wire / (hpwl.expand(B) + 0.1)                  # budget / tree size
        w4 = clust_dia * knn4_mean.expand(B)                     # scope × local spacing
        w5 = max_wire * clust_sz                                  # segment × n_segments
        wl_int = self.ln_wl_int(
            torch.stack([w1, w2, w3, w4, w5], dim=1))  # [B, 5]

        # ── Task heads ────────────────────────────────────────────────────
        phi = torch.cat([enc_exp, knobs_batch], dim=1)   # [B, 64+4]

        # Skew head gets global enc + per-FF order stats + knobs + interactions
        phi_skew = torch.cat([enc_exp, skew_enc_exp, knobs_batch, skew_int], dim=1)
        y_skew  = self.head_skew(phi_skew).squeeze(-1)

        # Power and WL use global features only
        y_power = self.head_power(torch.cat([phi, power_int], dim=1)).squeeze(-1)
        y_wl    = self.head_wl   (torch.cat([phi, wl_int],    dim=1)).squeeze(-1)

        return {'skew': y_skew, 'power': y_power, 'wl': y_wl}


# ─────────────────────────────────────────────────────────────────────────────
# 5. TRAINING
# ─────────────────────────────────────────────────────────────────────────────

def run_placement(model, pl, device, task_weights=None):
    feats  = pl['ff_feats'].to(device)
    ff_X   = torch.tensor(pl['ff_X_smooth'], dtype=torch.float32).to(device)              if not isinstance(pl['ff_X_smooth'], torch.Tensor)              else pl['ff_X_smooth'].to(device)
    skip   = pl['ff_skip_local'].to(device)
    knobs  = torch.stack([c['knobs'].to(device) for c in pl['cts_runs']])  # [B,4]
    t_skew = torch.stack([c['targets']['skew'].to(device).squeeze()
                          for c in pl['cts_runs']])
    t_pow  = torch.stack([c['targets']['power'].to(device).squeeze()
                          for c in pl['cts_runs']])
    t_wl   = torch.stack([c['targets']['wl'].to(device).squeeze()
                          for c in pl['cts_runs']])

    preds = model(feats, ff_X, skip, knobs)

    if task_weights is None:
        task_weights = {'skew': 1.5, 'power': 1.0, 'wl': 1.0}

    l_s = F.l1_loss(preds['skew'],  t_skew)
    l_p = F.l1_loss(preds['power'], t_pow)
    l_w = F.l1_loss(preds['wl'],    t_wl)

    total = task_weights['skew']*l_s + task_weights['power']*l_p + task_weights['wl']*l_w

    return (total,
            {t: preds[t].detach().cpu().tolist() for t in ['skew','power','wl']},
            {'skew': t_skew.cpu().tolist(),
             'power':t_pow.cpu().tolist(),
             'wl':   t_wl.cpu().tolist()},
            {'skew': l_s.item(), 'power': l_p.item(), 'wl': l_w.item()})


def evaluate(model, placements, device):
    model.eval()
    all_p = defaultdict(list); all_t = defaultdict(list)
    with torch.no_grad():
        for pl in placements:
            _, preds, targets, _ = run_placement(model, pl, device)
            for task in ['skew','power','wl']:
                all_p[task].extend(preds[task])
                all_t[task].extend(targets[task])
    return {task: float(np.abs(np.array(all_p[task]) -
                               np.array(all_t[task])).mean())
            for task in ['skew','power','wl']}


def train_and_evaluate(placements, device):
    print(f"\n{'═'*60}")
    print(f"Leave-one-placement-out CV  |  {len(placements)} placements")
    print(f"Features: {FEAT_DIM}  EPOCHS: {EPOCHS}  LR: {LR}")
    print(f"{'═'*60}")

    all_results = defaultdict(lambda: defaultdict(list))

    for hold_idx, held in enumerate(placements):
        train_set = [p for i,p in enumerate(placements) if i != hold_idx]
        design    = held['design_name']

        print(f"\n[{hold_idx+1}/{len(placements)}]  "
              f"held={held['placement_id']}  design={design}  "
              f"n_ff={held['n_ff']}  train_n={len(train_set)}")

        model = CTSPredictor(feat_dim=FEAT_DIM, sgc_dim=SGC_DIM,
                             n_knobs=NUM_KNOBS, hidden=HIDDEN).to(device)
        n_params = sum(p.numel() for p in model.parameters())
        print(f"  params={n_params:,}")

        optimizer  = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=LR*0.1)

        best_val   = float('inf')
        best_state = None

        # GradNorm: slow adaptation so it doesn't chase per-placement noise
        task_weights   = {'skew': 1.5, 'power': 1.0, 'wl': 1.0}
        loss_ema       = {'skew': None, 'power': None, 'wl': None}
        GRADNORM_ALPHA = 0.03

        for epoch in range(EPOCHS):
            model.train()
            random.shuffle(train_set)
            epoch_loss = 0.

            for pl in train_set:
                optimizer.zero_grad()
                total, _, _, per_task = run_placement(
                    model, pl, device, task_weights)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += total.item()

            scheduler.step()

            # GradNorm update
            for t in ['skew','power','wl']:
                l = per_task[t]
                loss_ema[t] = l if loss_ema[t] is None \
                              else 0.9*loss_ema[t] + 0.1*l
            total_l = sum(loss_ema[t]+1e-8 for t in ['skew','power','wl'])
            for t in ['skew','power','wl']:
                target_w = (loss_ema[t]+1e-8) / total_l * 3.0
                task_weights[t] = ((1-GRADNORM_ALPHA)*task_weights[t]
                                   + GRADNORM_ALPHA*target_w)

            if epoch % 50 == 0 or epoch == EPOCHS-1:
                val  = evaluate(model, [held], device)
                # max() prevents whack-a-mole
                score = max(val.values())
                flag  = ''
                if score < best_val:
                    best_val   = score
                    best_state = {k: v.cpu().clone()
                                  for k,v in model.state_dict().items()}
                    flag = '  ← best'
                lr_now = scheduler.get_last_lr()[0]
                print(f"  ep{epoch:3d} | "
                      f"train={epoch_loss/len(train_set):.4f} | "
                      f"skew={val['skew']:.4f}  "
                      f"power={val['power']:.4f}  "
                      f"wl={val['wl']:.4f}  "
                      f"lr={lr_now:.5f}{flag}")

        if best_state:
            model.load_state_dict({k: v.to(device)
                                   for k,v in best_state.items()})
        final = evaluate(model, [held], device)
        print(f"\n  ── FINAL  {held['placement_id']} ──")
        for task in ['skew','power','wl']:
            print(f"     {task:6s}: {final[task]:.4f}")
        for task, mae in final.items():
            all_results[design][task].append(mae)

    # Summary
    print(f"\n{'═'*60}\nRESULTS (per-placement normalized MAE)\n{'═'*60}")
    all_maes = defaultdict(list)
    for design, tm in sorted(all_results.items()):
        print(f"\n  {design}:")
        for task in ['skew','power','wl']:
            vals = tm[task]
            print(f"    {task:6s}: {np.mean(vals):.4f} ± "
                  f"{np.std(vals):.4f}  (n={len(vals)})")
            all_maes[task].extend(vals)
    print(f"\n  OVERALL:")
    for task in ['skew','power','wl']:
        print(f"    {task:6s}: {np.mean(all_maes[task]):.4f}")
    print(f"{'═'*60}\n")
    return all_results


def train_and_evaluate_lodo(placements, device):
    """
    Leave-One-Design-Out cross-validation.
    Train on 3 designs, test on the 4th.
    This is the true generalization test — the held-out design was
    never seen during training in any form.
    """
    designs = sorted(set(p['design_name'] for p in placements))
    print(f"\n{'═'*60}")
    print(f"Leave-One-Design-Out CV  |  {len(designs)} designs")
    print(f"{'═'*60}")

    lodo_results = {}

    for held_design in designs:
        train_set = [p for p in placements if p['design_name'] != held_design]
        test_set  = [p for p in placements if p['design_name'] == held_design]

        print(f"\n[LODO]  held_design={held_design}  "
              f"train_n={len(train_set)}  test_n={len(test_set)}")

        model = CTSPredictor(feat_dim=FEAT_DIM, sgc_dim=SGC_DIM,
                             n_knobs=NUM_KNOBS, hidden=HIDDEN).to(device)
        print(f"  params={sum(p.numel() for p in model.parameters()):,}")

        optimizer = Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS, eta_min=LR*0.1)

        task_weights   = {'skew': 1.5, 'power': 1.0, 'wl': 1.0}
        loss_ema       = {'skew': None, 'power': None, 'wl': None}
        GRADNORM_ALPHA = 0.03
        best_val   = float('inf')
        best_state = None

        for epoch in range(EPOCHS):
            model.train()
            random.shuffle(train_set)
            epoch_loss = 0.

            for pl in train_set:
                optimizer.zero_grad()
                total, _, _, per_task = run_placement(model, pl, device, task_weights)
                total.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                epoch_loss += total.item()
            scheduler.step()

            for t in ['skew','power','wl']:
                l = per_task[t]
                loss_ema[t] = l if loss_ema[t] is None else 0.9*loss_ema[t] + 0.1*l
            total_l = sum(loss_ema[t]+1e-8 for t in ['skew','power','wl'])
            for t in ['skew','power','wl']:
                target_w = (loss_ema[t]+1e-8) / total_l * 3.0
                task_weights[t] = (1-GRADNORM_ALPHA)*task_weights[t] + GRADNORM_ALPHA*target_w

            if epoch % 50 == 0 or epoch == EPOCHS-1:
                val  = evaluate(model, test_set, device)
                score = max(val.values())
                flag  = ''
                if score < best_val:
                    best_val   = score
                    best_state = {k: v.cpu().clone() for k,v in model.state_dict().items()}
                    flag = '  ← best'
                print(f"  ep{epoch:3d} | train={epoch_loss/len(train_set):.4f} | "
                      f"skew={val['skew']:.4f}  power={val['power']:.4f}  "
                      f"wl={val['wl']:.4f}{flag}")

        if best_state:
            model.load_state_dict({k: v.to(device) for k,v in best_state.items()})
        final = evaluate(model, test_set, device)
        print(f"\n  ── FINAL on {held_design} (UNSEEN) ──")
        for task in ['skew','power','wl']:
            print(f"     {task:6s}: {final[task]:.4f}")
        lodo_results[held_design] = final

    print(f"\n{'═'*60}\nLODO RESULTS (true generalization)\n{'═'*60}")
    for design, res in sorted(lodo_results.items()):
        print(f"  {design}:  skew={res['skew']:.4f}  "
              f"power={res['power']:.4f}  wl={res['wl']:.4f}")
    for task in ['skew','power','wl']:
        mean = np.mean([lodo_results[d][task] for d in lodo_results])
        print(f"  MEAN {task:6s}: {mean:.4f}")
    print(f"{'═'*60}\n")
    return lodo_results


# ─────────────────────────────────────────────────────────────────────────────
# 6. MAIN
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == '__main__':
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}\nPT:     {PT_DIR}\nCSV:    {CSV_PATH}\n")

    result = load_dataset(PT_DIR, CSV_PATH)
    if not result:
        print("ERROR: No placements loaded."); exit(1)

    placements, mu_ff, sig_ff, mu_sgc, sig_sgc = result

    print(f"\nModel dimensions:")
    print(f"  Physics features: {FEAT_DIM}")
    print(f"  Knobs:            {NUM_KNOBS}")
    print(f"  Skew interactions: {CTSPredictor.N_SKEW_INT}")
    print(f"  Power interactions:{CTSPredictor.N_POWER_INT}")
    print(f"  WL interactions:   {CTSPredictor.N_WL_INT}")

    # Leave-one-placement-out: standard CV
    train_and_evaluate(placements, device)

    # Leave-one-design-out: true generalization test
    print("\n" + "="*60)
    print("Running Leave-One-Design-Out for generalization evaluation...")
    train_and_evaluate_lodo(placements, device)

Device: cuda
PT:     processed_graphs/
CSV:    dataset_with_def/unified_manifest_normalized.csv

Found 140 .pt files

────────────────────────────────────────────────
Loading: aes_run_20260306_173215  (aes)


/home/rain/CTS-Task-Aware-Clustering/venv/lib/python3.12/site-packages/torch/_utils.py:361: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:49.)
  result = torch.sparse_compressed_tensor(


  OK  n_ff=2994  skip=5595  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_175900  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_180433  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_181625  (aes)
  OK  n_ff=2994  skip=5595  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_183245  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_190630  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_194107  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_195250  (aes)
  OK  n_ff=2994  skip=5596  runs=10

────────────────────────────────────────────────
Loading: aes_run_20260306_200621

KeyboardInterrupt: 